# Evidence Detection (ED) — DeBERTa v3 baseline
Binary classification for (claim, evidence) pairs using fine-tuned DeBERTa v3.

## Install and Imports

In [1]:
# In a fresh environment install the required packages first:
# pip install pandas numpy scikit-learn torch transformers datasets accelerate sentencepiece protobuf tiktoken
import os, re, random, json
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)


def has_mps() -> bool:
    return (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_built()
        and torch.backends.mps.is_available()
    )


DEVICE = "cuda" if torch.cuda.is_available() else "mps" if has_mps() else "cpu"
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, matthews_corrcoef

# for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Using device:", DEVICE)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model Training

### Config and load data

In [ ]:
CONFIG = {
    "model_name": "microsoft/deberta-v3-base",
    "max_length": 256,
    "train_bs": 16,
    "eval_bs": 32,
    "infer_eval_bs": 64,
    "grad_accum": 1,
    "lr": 2e-5,
    "epochs": 3,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "lr_scheduler_type": "linear",
    "out_dir": "./outputs/deberta_v3_baseline_plain_trainer",
    "save_dir": "./outputs/deberta_v3_baseline_plain_trainer/best_model",
    "train_path": "./training_data/train.csv",
    "dev_path": "./training_data/dev.csv",
}


def _env_flag(name, default=False):
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}


def _env_int(name, default):
    raw = os.getenv(name)
    return int(raw) if raw not in (None, "") else default


def _env_float(name, default):
    raw = os.getenv(name)
    return float(raw) if raw not in (None, "") else default


TEST_MODE = _env_flag("SOLUTION_C_TEST_MODE", False)
CONFIG["epochs"] = _env_int("SOLUTION_C_EPOCHS", CONFIG["epochs"])
CONFIG["lr"] = _env_float("SOLUTION_C_LR", CONFIG["lr"])
CONFIG["weight_decay"] = _env_float("SOLUTION_C_WEIGHT_DECAY", CONFIG["weight_decay"])
CONFIG["lr_scheduler_type"] = os.getenv("SOLUTION_C_LR_SCHEDULER", CONFIG["lr_scheduler_type"])
CONFIG["warmup_ratio"] = _env_float("SOLUTION_C_WARMUP_RATIO", CONFIG["warmup_ratio"])
CONFIG["max_length"] = _env_int("SOLUTION_C_MAX_LENGTH", CONFIG["max_length"])
CONFIG["train_bs"] = _env_int("SOLUTION_C_TRAIN_BS", CONFIG["train_bs"])
CONFIG["eval_bs"] = _env_int("SOLUTION_C_EVAL_BS", CONFIG["eval_bs"])
CONFIG["infer_eval_bs"] = _env_int("SOLUTION_C_INFER_EVAL_BS", CONFIG["infer_eval_bs"])
CONFIG["grad_accum"] = _env_int("SOLUTION_C_GRAD_ACCUM", CONFIG["grad_accum"])
CONFIG["use_class_weights"] = _env_flag("SOLUTION_C_USE_CLASS_WEIGHTS", False)
CONFIG["out_dir"] = os.getenv("SOLUTION_C_OUT_DIR", CONFIG["out_dir"])
CONFIG["save_dir"] = os.getenv("SOLUTION_C_SAVE_DIR", f"{CONFIG['out_dir']}/best_model")

if DEVICE == "mps":
    CONFIG["max_length"] = 192
    CONFIG["train_bs"] = 2
    CONFIG["eval_bs"] = 4
    CONFIG["infer_eval_bs"] = 8
    CONFIG["grad_accum"] = 8
elif DEVICE == "cpu":
    CONFIG["train_bs"] = 4
    CONFIG["eval_bs"] = 8
    CONFIG["infer_eval_bs"] = 16
    CONFIG["grad_accum"] = 4

# load data into dataframes
train_df = pd.read_csv(CONFIG["train_path"])
dev_df   = pd.read_csv(CONFIG["dev_path"])

# ensure labels are integers
train_df["label"] = train_df["label"].astype(int)
dev_df["label"]   = dev_df["label"].astype(int)

if TEST_MODE:
    train_df = train_df.sample(128, random_state=SEED).reset_index(drop=True)
    dev_df   = dev_df.sample(128, random_state=SEED).reset_index(drop=True)

train_label_counts = train_df["label"].value_counts().sort_index().reindex([0, 1], fill_value=0)
class_weights = None
if CONFIG["use_class_weights"]:
    total_examples = float(train_label_counts.sum())
    class_weights = torch.tensor(
        [total_examples / (2.0 * max(int(train_label_counts.loc[label]), 1)) for label in [0, 1]],
        dtype=torch.float32,
    )

# print dataset stats
print("Train:", len(train_df), "Dev:", len(dev_df))
print("Train label dist:", train_df["label"].value_counts().to_dict())
print("Dev label dist:", dev_df["label"].value_counts().to_dict())
print("Runtime config:", {k: CONFIG[k] for k in ["train_bs", "eval_bs", "infer_eval_bs", "grad_accum", "max_length", "epochs", "lr", "weight_decay", "lr_scheduler_type", "use_class_weights", "out_dir", "save_dir"]})
if class_weights is not None:
    print("Class weights:", class_weights.tolist())
train_df.head(3)


Train: 128 Dev: 128
Train label dist: {0: 90, 1: 38}
Dev label dist: {0: 93, 1: 35}


,Claim,Evidence,label
0,We should abolish zoos,The zoo endeavors to create an ecologically-fr...,0
1,We should disband NASA,discovered NASA's open source code and contact...,0
2,We should subsidize journalism,"As the movement launched, Shao widely spread t...",0


### Tokenisation

In [3]:
# create datasets and tokeniser
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=False)
train_ds = Dataset.from_pandas(train_df[["Claim", "Evidence", "label"]], preserve_index=False)
dev_ds   = Dataset.from_pandas(dev_df[["Claim", "Evidence", "label"]], preserve_index=False)

# tokenize the datasets
def tok_fn(batch):
    return tokenizer(
        batch["Claim"],
        batch["Evidence"],
        truncation=True,
        max_length=CONFIG["max_length"],
    )
train_tok = train_ds.map(tok_fn, batched=True, remove_columns=["Claim", "Evidence"])
dev_tok   = dev_ds.map(tok_fn, batched=True, remove_columns=["Claim", "Evidence"])

# pads inputs to the same length in a batch
collator = DataCollatorWithPadding(tokenizer=tokenizer)


Map: 100%|██████████| 128/128 [00:00<00:00, 22774.82 examples/s]


### Training and metrics

In [10]:
# metric function that trainer calls on evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    bin_p, bin_r, bin_f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    mcc = matthews_corrcoef(labels, preds)
    return {
        "accuracy": acc,
        "precision": bin_p,
        "recall": bin_r,
        "f1": bin_f1,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "mcc": mcc,
    }


# load pretrained DeBERTa v3 with a classification head for 2 labels
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=2,
    id2label={0: "not_relevant", 1: "relevant"},
    label2id={"not_relevant": 0, "relevant": 1},
    torch_dtype=torch.float32,
)
model.to(dtype=torch.float32)

if DEVICE != "cuda":
    model.gradient_checkpointing_enable()

# controls training/eval, checkpointing, and optimisation hyperparameters.
args = TrainingArguments(
    output_dir=CONFIG["out_dir"],
    learning_rate=CONFIG["lr"],
    per_device_train_batch_size=CONFIG["train_bs"],
    per_device_eval_batch_size=CONFIG["eval_bs"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    max_steps=2 if TEST_MODE else -1,
    num_train_epochs=CONFIG["epochs"] if not TEST_MODE else 1,
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"] if not TEST_MODE else 0.0,
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    eval_strategy="epoch",
    save_strategy="epoch" if not TEST_MODE else "steps",
    save_steps=1 if TEST_MODE else None,
    load_best_model_at_end=True if not TEST_MODE else False,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2 if not TEST_MODE else 1,
    gradient_checkpointing=DEVICE != "cuda",
    fp16=False,
    optim="adamw_torch",
    adam_epsilon=1e-6,
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    disable_tqdm=True,
)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if self.class_weights is None:
            return super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)

        model_inputs = dict(inputs)
        labels = model_inputs.pop("labels", None)
        if labels is None:
            labels = model_inputs.pop("label")
        model_inputs["labels"] = labels

        outputs = model(**model_inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(device=logits.device, dtype=logits.dtype)
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1).long())
        return (loss, outputs) if return_outputs else loss


# full training loop
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

# Train the model (fine-tuning).
train_result = trainer.train()
print("Train metrics:", train_result.metrics)

# Evaluate the best checkpoint on validation set.
eval_metrics = trainer.evaluate()
print("Eval metrics:", eval_metrics)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8890.59it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint

{'eval_loss': '0.7062', 'eval_accuracy': '0.625', 'eval_precision': '0', 'eval_recall': '0', 'eval_f1': '0', 'eval_runtime': '0.7247', 'eval_samples_per_second': '176.6', 'eval_steps_per_second': '5.519', 'epoch': '0.25'}
{'train_runtime': '11.09', 'train_samples_per_second': '2.884', 'train_steps_per_second': '0.18', 'train_loss': '0.721', 'epoch': '0.25'}


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': '0.7062', 'eval_accuracy': '0.625', 'eval_precision': '0', 'eval_recall': '0', 'eval_f1': '0', 'eval_runtime': '0.6651', 'eval_samples_per_second': '192.5', 'eval_steps_per_second': '6.014', 'epoch': '0.25'}


{'eval_loss': 0.7062010169029236,
 'eval_accuracy': 0.625,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_runtime': 0.6651,
 'eval_samples_per_second': 192.461,
 'eval_steps_per_second': 6.014,
 'epoch': 0.25}

### Save model

In [11]:
# Save the best model + tokenizer
os.makedirs(CONFIG["save_dir"], exist_ok=True)
trainer.save_model(CONFIG["save_dir"])
tokenizer.save_pretrained(CONFIG["save_dir"])

print("Saved model and tokenizer to:", CONFIG["save_dir"])

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.42it/s]

Saved model and tokenizer to: ./outputs/deberta_v3_baseline/best_model


## Model Demo

In [ ]:
# specify test file path and output path for predictions
TEST_PATH = "./test.csv"
OUT_PATH  = "./predictions.csv"

@torch.inference_mode()  # disables gradient tracking to make inference faster/cheaper
def predict_test(model_dir: str, test_csv: str, out_csv: str) -> pd.DataFrame:
    # load fine-tuned model + tokenizer from disk
    tok = AutoTokenizer.from_pretrained(model_dir, use_fast=False)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir, torch_dtype=torch.float32)
    mdl.to(dtype=torch.float32)

    # put model in eval mode
    mdl.eval()

    # move model to selected accelerator
    mdl.to(DEVICE)

    # Load test file 
    df = pd.read_csv(test_csv)

    # convert to HF dataset 
    ds = Dataset.from_pandas(df[["Claim", "Evidence"]], preserve_index=False)

    # Tokenise pairs in same way as training
    def tok_fn(batch):
        return tok(batch["Claim"], batch["Evidence"], truncation=True, max_length=CONFIG["max_length"])

    ds_tok = ds.map(tok_fn, batched=True, remove_columns=["Claim", "Evidence"])

    # pad batches
    collator = DataCollatorWithPadding(tokenizer=tok)

    # minimal Trainer for batched prediction
    infer_args = TrainingArguments(
        output_dir=os.path.join(model_dir, "_infer_tmp"), 
        per_device_eval_batch_size=CONFIG["infer_eval_bs"],
        dataloader_pin_memory=torch.cuda.is_available(),
        report_to="none",
    )
    infer_trainer = Trainer(model=mdl, args=infer_args, processing_class=tok, data_collator=collator)

    # returns logits for each example
    pred = infer_trainer.predict(ds_tok)

    # convert logits to predicted class
    yhat = np.argmax(pred.predictions, axis=-1)

    # write output CSV
    out = df.copy()
    out["pred"] = yhat.astype(int)
    out.to_csv(out_csv, index=False)

    return out

# Run demo inference and preview results
demo = predict_test(CONFIG["save_dir"], TEST_PATH, OUT_PATH)
demo.head(5), print("Wrote:", OUT_PATH)
